In [49]:
from utils import variables, functions

# realod them (maybe have been updated)
import importlib
importlib.reload(functions)
importlib.reload(variables)

import numpy as np, os, json, pandas as pd, random, pickle, math, re
from IPython.display import clear_output
import networkx as nx
import itertools
import joblib

# Matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.patches import Rectangle
import matplotlib.patches as mpatches

# Torch
import torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, DataLoader

# Sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import RidgeCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import ParameterGrid

Functions imported correctly.
Variables imported correctly.


To do:
- provare CNN con tutti i dati
- problema: numnero di voxels totali diversi per soggetto
- soluzione: riduzione di dimensionalità (tipo da 1685, 212, ..., ... a 1000 ad esempio.)
- ma senza dividerli per ROI come ho fatto finora con transformer, BRNN, e gcn

In [25]:
# Load all the dataframes
confounds_df = pd.read_csv(os.path.join(variables.BOLD5000_PATH,'confounds_df.csv'))
rois_df = pd.read_csv(os.path.join(variables.BOLD5000_PATH,'rois_df.csv'))
functionals_df = pd.read_csv(os.path.join(variables.BOLD5000_PATH,'functionals_df.csv'))
events_df = pd.read_csv(os.path.join(variables.BOLD5000_PATH,'events_df.csv'))
trials_df = pd.read_csv(os.path.join(variables.BOLD5000_PATH,'trials_df.csv'))

In [26]:
# Load the files
subjects_data = []
subjects_labels = []
for sub in ['1','2','3','4']:

    if os.path.exists(os.path.join(variables.BOLD5000_PATH, f'subject{sub}_features.pkl')):
        with open(os.path.join(variables.BOLD5000_PATH, f'subject{sub}_features.pkl'), "rb") as fp:
            data = pickle.load(fp)

        # Extract the elements
        f = data["features"]
        l = data["labels"]
        n = data["n_nans"]

        print(f"Number of runs for subject {sub}:", len(f))
        print(f"Number of labels for subject {sub} and first run:", len(l[0]),'\n')
        #print("Number of NaNs:", n)

        stacked_trials = []
        stacked_labels = []
        for trial, lab in zip(f,l):
            if trial is not None:
                stacked_trials.extend(trial)    # adds all the runs of a subject
                stacked_labels.extend(lab)

        subjects_data.append(stacked_trials)
        subjects_labels.append(stacked_labels)

    else:
        subjects_data.append(None)
        subjects_labels.append(None)

print(f'Len subjects_data: {len(subjects_data)}\n')
for i in range(len(subjects_data)):
    if subjects_data[i] is not None:
        print(f'Subject {i+1}')
        print(f'Type subjects_data[{i}]: {type(subjects_data[i])} - Len: {len(subjects_data[i])}')
        print(f'Type subjects_data[{i}[0]]: {type(subjects_data[i][0])} - Len: {len(subjects_data[i][0])}\n')

Number of runs for subject 1: 142
Number of labels for subject 1 and first run: 22 

Number of runs for subject 2: 142
Number of labels for subject 2 and first run: 21 

Number of runs for subject 3: 142
Number of labels for subject 3 and first run: 22 

Number of runs for subject 4: 84
Number of labels for subject 4 and first run: 21 

Len subjects_data: 4

Subject 1
Type subjects_data[0]: <class 'list'> - Len: 3119
Type subjects_data[0[0]]: <class 'list'> - Len: 1685

Subject 2
Type subjects_data[1]: <class 'list'> - Len: 3119
Type subjects_data[1[0]]: <class 'list'> - Len: 2270

Subject 3
Type subjects_data[2]: <class 'list'> - Len: 3121
Type subjects_data[2[0]]: <class 'list'> - Len: 3104

Subject 4
Type subjects_data[3]: <class 'list'> - Len: 1834
Type subjects_data[3[0]]: <class 'list'> - Len: 2787



In [38]:
X_train_all, y_train_all = [], []
X_val_all, y_val_all = [], []
X_test_all, y_test_all = [], []

for X_sub, y_sub in zip(subjects_data, subjects_labels):
    if X_sub is not None:

        # Scaling for subject
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_sub)

        # Train/test split
        X_temp, X_test, y_temp, y_test = train_test_split(X_scaled, y_sub, test_size=0.3, stratify=y_sub, random_state=42)

        # train/val split
        X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=42)

        # PCA to reduce dimensionality
        pca = PCA(n_components=1024, random_state=42)
        X_pca = pca.fit_transform(X_train)

        X_train_all.append(X_pca)
        y_train_all.append(y_train)

        X_val_all.append(pca.transform(X_val))
        y_val_all.append(y_val)

        X_test_all.append(pca.transform(X_test))
        y_test_all.append(y_test)

# Concatenate
X_train_all = np.vstack(X_train_all)
y_train_all = np.hstack(y_train_all)

X_val_all = np.vstack(X_val_all)
y_val_all = np.hstack(y_val_all)

X_test_all = np.vstack(X_test_all)
y_test_all = np.hstack(y_test_all)

In [39]:
print("Train set:")
print("  X_train_all shape:", X_train_all.shape)
print("  y_train_all shape:", y_train_all.shape)

print("\nValidation set:")
print("  X_val_all shape:", X_val_all.shape)
print("  y_val_all shape:", y_val_all.shape)

print("\nTest set:")
print("  X_test_all shape:", X_test_all.shape)
print("  y_test_all shape:", y_test_all.shape)

Train set:
  X_train_all shape: (6265, 1024)
  y_train_all shape: (6265,)

Validation set:
  X_val_all shape: (1568, 1024)
  y_val_all shape: (1568,)

Test set:
  X_test_all shape: (3360, 1024)
  y_test_all shape: (3360,)


In [50]:
def train_grid_search_mlp(grid_params, X_train, y_train, X_val, y_val):

    # Since the class are unbalanced I will chose the best model as the one with the highest F1 score (macro)
    best_params = None
    best_val_f1 = 0
    best_val_acc = 0
    
    for i, params in enumerate(ParameterGrid(grid_params)):
        print(f'Combination {i+1}/{len(ParameterGrid(grid_params))} - Params: {params}')

        # MLP classifier
        mlp = MLPClassifier(hidden_layer_sizes=params['hidden_layer_sizes'], activation=params['activation'], 
                            solver=params['solver'], alpha=params['alpha'], batch_size=params['batch_size'],
                            max_iter=100, random_state=42)
        mlp.fit(X_train, y_train)

        y_pred = mlp.predict(X_val)
        
        val_acc = accuracy_score(y_val, y_pred)
        val_f1 = f1_score(y_val, y_pred, average='weighted')

        # Save the best one
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_params = params
            best_val_acc = val_acc

        # Print metrics
        print(f" Val Acc: {val_acc:6.3f}  -  Val F1: {val_f1:6.3f}\n")

    return best_params, best_val_f1, best_val_acc

In [51]:
grid_params = {
    'hidden_layer_sizes': [(512, 256, 128, 64), (256, 128, 64), (256, 64), (128, 64)],
    'activation': ['relu', 'logistic', 'tanh'],
    'solver': ['sgd', 'adam'],
    'alpha': [0.00001, 0.0001, 0.001, 0.01],
    'batch_size': [32, 64, 128]
}

print(f'Number of combinations: {len(ParameterGrid(grid_params))}\n')

device = "cuda" if torch.cuda.is_available() else "cpu"
if not os.path.exists(os.path.join(variables.BOLD5000_PATH, 'best_params_mlp.txt')):
    best_params, best_val_f1, best_val_acc = train_grid_search_mlp(grid_params, X_train_all, y_train_all, X_val_all, y_val_all)
    with open(os.path.join(variables.BOLD5000_PATH, 'best_params_mlp.txt'), "w") as f: 
        json.dump(best_params, f, indent=4)
else: # Load best params 
    with open(os.path.join(variables.BOLD5000_PATH, 'best_params_mlp.txt'), "r") as f: best_params = eval(f.read()) 
    print("Loaded best MLP parameters:", best_params)

Number of combinations: 288

Loaded best MLP parameters: {'activation': 'logistic', 'alpha': 0.01, 'batch_size': 32, 'hidden_layer_sizes': [256, 64], 'solver': 'adam'}


In [52]:
print(f'Best params: {best_params}')

Best params: {'activation': 'logistic', 'alpha': 0.01, 'batch_size': 32, 'hidden_layer_sizes': [256, 64], 'solver': 'adam'}


In [53]:
# Load the best params saved
if os.path.exists(os.path.join(variables.BOLD5000_PATH, 'best_params_mlp.txt')):
    with open(os.path.join(variables.BOLD5000_PATH, 'best_params_mlp.txt'), 'r') as file:
        best_params_string = json.load(file)

# from strings to original type
best_params = {}
for i in best_params_string.keys():
    if i == 'lr' or i == 'alpha':
        best_params[i] = float(best_params_string[i])
    elif i == 'hidden_layer_sizes':
        best_params[i] = eval(f"[{best_params_string[i]}]")
    elif i == 'batch_size':
        best_params[i] = int(best_params_string[i])
    else:
        best_params[i] = best_params_string[i]

print(f'Best params: {best_params}')

Best params: {'activation': 'logistic', 'alpha': 0.01, 'batch_size': 32, 'hidden_layer_sizes': [[256, 64]], 'solver': 'adam'}


In [54]:
def train_best_mlp(best_params, X_train, y_train, X_val, y_val, device):

    for i, params in enumerate(ParameterGrid(best_params)):
        print(f'Combination {i+1}/{len(ParameterGrid(best_params))} - Params: {params}')

        # MLP classifier
        mlp = MLPClassifier(hidden_layer_sizes=params['hidden_layer_sizes'], activation=params['activation'], 
                            solver=params['solver'], alpha=params['alpha'], batch_size=params['batch_size'],
                            max_iter=1000, random_state=42)
        mlp.fit(X_train, y_train)

        y_pred = mlp.predict(X_val)
        
        val_acc = accuracy_score(y_val, y_pred)
        val_f1 = f1_score(y_val, y_pred, average="macro")

        joblib.dump(mlp, "/workspace/devcontainer-bigdata/models/best_model_mlp.pkl") 

        # Print metrics
        print(f" Val Acc: {val_acc:6.3f}  -  Val F1: {val_f1:6.3f}\n")   

In [ ]:
best_params = {
    "activation": ["logistic"],
    "alpha": [0.01],
    "batch_size": [32],
    "hidden_layer_sizes": [[256, 64]],
    "solver": ["adam"]
}

In [57]:
device = "cuda" if torch.cuda.is_available() else "cpu"
train_best_mlp(best_params, X_train_all, y_train_all, X_val_all, y_val_all, device)

Combination 1/1 - Params: {'activation': 'logistic', 'alpha': 0.01, 'batch_size': 32, 'hidden_layer_sizes': [256, 64], 'solver': 'adam'}
 Val Acc:  0.439  -  Val F1:  0.246



In [58]:
mlp = joblib.load("/workspace/devcontainer-bigdata/models/best_model_mlp.pkl")

In [61]:
y_pred = mlp.predict(X_test_all)

acc  = accuracy_score(y_test_all, y_pred)
f1   = f1_score(y_test_all, y_pred, average="macro")
prec = precision_score(y_test_all, y_pred, average="macro")
rec  = recall_score(y_test_all, y_pred, average="macro")

print("Test Accuracy:", acc)
print("Test Precision:", prec)
print("Test Recall:", rec)
print("Test F1:", f1)

Test Accuracy: 0.44434523809523807
Test Precision: 0.26776859476838166
Test Recall: 0.25331631775714447
Test F1: 0.25448794742295644
